In [1]:
import os
import torch
import json
import numpy as np
import pandas as pd
from time import time
from tqdm import tqdm
import matplotlib.pyplot as plt
from multiprocessing import Pool

Header from original dataset

[year, month, day, hour, minute, lat, lon, wp, tir, size, mask]

In [2]:
start = time()

In [3]:
file = '/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/t1/shard-00000.pt'

data = torch.load(file)

input = data['inputs'].float()

# last column (11th) is the mask indicating whether it's a real core or a padded one
mask = input[:, :, 10] == 1

core = input[mask]

features = {
        "lats":  core[:, 5].numpy(),
        "lons":  core[:, 6].numpy(),
        "wps":   core[:, 7].numpy(),
        "tirs":  core[:, 8].numpy(),
        "sizes": core[:, 9].numpy(),
    }

In [9]:
# Compute scaling parameters
# log1p to stabilise range

scaling = {
    "lat_min":  float(features["lats"].min()),
    "lat_max":  float(features["lats"].max()),
    "lon_min":  float(features["lons"].min()),
    "lon_max":  float(features["lons"].max()),
    "wp_max":   float(np.log1p(features["wps"]).max()),  
    "tir_min":  float(features["tirs"].min()),
    "tir_max":  float(features["tirs"].max()),
    "size_max": float(np.log1p(features["sizes"].max())),
}

In [7]:
# ... your processing loop ...
print(f"Total time: {time() - start:.2f}s")

Total time: 18.44s


In [8]:
scaling

{'lat_min': -37.65168380737305,
 'lat_max': 23.619253158569336,
 'lon_min': -20.45563507080078,
 'lon_max': 54.89462661743164,
 'wp_max': 6.2110466957092285,
 'tir_min': -99.66666412353516,
 'tir_max': -45.0,
 'size_max': 11.494853019714355}

In [10]:
output_path = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/normalisation/parameters/normalisation.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    json.dump(scaling, f, indent=4)